---
title: "Discovering Dataset in the DestinE Data Lake (HDA)" 
subtitle: "This notebook shows how a new user can explore the DestinE Data Lake catalog and find the dataset (collection) IDs needed for any later data search or download.
"
author: "Author: Alejandro Fonseca (EUMETSAT/Starion)"
tags: [HDA,STAC,]
thumbnail: img/EUMETSAT_TEST.png 
license: MIT
copyright: "© 2024 EUMETSAT"
---

- **Objective:** Learn different ways to discover datasets: listing everything, searching by keyword, browsing by provider/information type, and filtering by useful attributes (space and time).
- **Data source:** The DEDL Harmonised Data Access (HDA) STAC API v2: `https://hda.data.destination-earth.eu/stac/v2`
- **Prerequisites:** Only the `pystac-client` package (`pip install pystac-client`). Dataset discovery does **not** require authentication or a DESP account.
- **Expected output:** Printed lists of collection IDs and titles matching each discovery method, plus the available search filters (queryables) for a chosen collection.

## How the catalog is organised (STAC in short)

The HDA API follows the [STAC standard](https://stacspec.org/) (SpatioTemporal Asset Catalog). Three concepts are enough to get started:

- **Catalog** : the entry point. One single URL that groups everything.
- **Collection** ; a dataset. For example, all Sentinel-2 L2A products form one collection. Each collection has a unique **ID**, a title, a description, keywords, and spatial/temporal extents.
- **Item** : a single piece of data inside a collection (one satellite scene, one model output slice), with its own datetime, geometry and download links (assets).

The workflow is always the same: first find the **collection ID**, then search for items inside it. This notebook covers the first step.

Collection IDs in HDA follow a naming convention that is itself useful for discovery:

```
EO.<PROVIDER>.DAT.<DATASET_NAME>
```

For example `EO.ESA.DAT.SENTINEL-2.MSI.L2A` (ESA) or `EO.ECMWF.DAT.DT_CLIMATE_ADAPTATION` (ECMWF) or 'EO.EUM.DAT.SENTINEL-3.OL_2_WFR' (EUMETSAT)

## Discover

In [16]:
pip install --user --quiet --upgrade destinelab

Note: you may need to restart the kernel to use updated packages.


In [14]:
import pystac_client

## Connect to the HDA STAC catalog

No token or credentials are needed for discovery.

In [2]:
HDA_STAC_URL = "https://hda.data.destination-earth.eu/stac/v2"

catalog = pystac_client.Client.open(HDA_STAC_URL)

print(f"Connected to: {catalog.title}")
print(f"Description: {catalog.description}")

Connected to: DEDL HDA STAC API
Description: STAC API for the DEDL Harmonized Data Access


## 1. List all available collections

The most direct way to see what exists: ask the catalog for every collection and print its ID and title. The ID is the value you will reuse everywhere else.

In [3]:
collections = list(catalog.get_collections())

print(f"Total collections: {len(collections)}\n")

for collection in collections:
    print(f"{collection.id}  —  {collection.title}")

Total collections: 311

EO.ECMWF.DAT.D1.DT_CLIMATE.G1.CMIP6_HIST_ICON.R1  —  Climate Change Adaptation Digital Twin (Climate Adaptation DT) - Historical Simulation - ICON - Generation-1 - Realization-1
EO.ECMWF.DAT.D1.DT_CLIMATE.G1.CMIP6_HIST_IFS-NEMO.R1  —  Climate Change Adaptation Digital Twin (Climate Adaptation DT) - Historical Simulation - IFS-NEMO - Generation-1 - Realization-1
EO.ECMWF.DAT.D1.DT_CLIMATE.G1.HIGHRESMIP_CONT_IFS-FESOM.R1  —  Climate Change Adaptation Digital Twin (Climate Adaptation DT) - Control Simulation - IFS-FESOM - Generation-1 - Realization-1
EO.ECMWF.DAT.D1.DT_CLIMATE.G1.HIGHRESMIP_CONT_IFS-NEMO.R1  —  Climate Change Adaptation Digital Twin (Climate Adaptation DT) - Control Simulation - IFS-NEMO - Generation-1 - Realization-1
EO.ECMWF.DAT.D1.DT_CLIMATE.G1.SCENARIOMIP_SSP3-7.0_ICON.R1  —  Climate Change Adaptation Digital Twin (Climate Adaptation DT) - Future Projection - ICON - Generation-1 - Realization-1
EO.ECMWF.DAT.D1.DT_CLIMATE.G1.SCENARIOMIP_SSP3-7.0

## 2. Search by keyword (free text)

Scrolling through 200+ collections is not practical. The `q` parameter of `collection_search` runs a free-text search over collection titles, descriptions and keywords.

- A single term matches collections containing that term.
- Comma-separated terms act as an OR: any of the terms may match.
- Quoted text matches the exact phrase.

In [ ]:
def print_results(results, label):
    print(f"{label}: {len(results)} collections found")
    for collection in results:
        print(f"  {collection.id}  —  {collection.title}")
    print()


# Single term
results = catalog.collection_search(q="temperature").collection_list()
print_results(results, "q='temperature'")

q='temperature': 49 collections found
  EO.ECMWF.DAT.ERA5_LAND_HOURLY  —  ERA5-Land hourly data from 1950 to present
  EO.MO.DAT.SST_GLO_SST_L4_NRT_OBSERVATIONS_010_001  —  Global Ocean OSTIA Sea Surface Temperature and Sea Ice Analysis
  EO.EUM.DAT.METOP.IASTHR011  —  IASI All Sky Temperature and Humidity Profiles - Climate Data Record Release 1.1 - Metop-A and -B
  EO.MO.DAT.SST_GLO_SST_L4_REP_OBSERVATIONS_010_011  —  Global Ocean OSTIA Sea Surface Temperature and Sea Ice Reprocessed
  EO.MO.DAT.SST_GLO_SST_L4_REP_OBSERVATIONS_010_024  —  ESA SST CCI and C3S reprocessed sea surface temperature analyses
  EO.EUM.DAT.METOP.IASSND02  —  IASI Combined Sounding Products - Metop
  EO.MO.DAT.SST_GLO_SST_L3S_NRT_OBSERVATIONS_010_010  —  ODYSSEA Global Ocean - Sea Surface Temperature Multi-sensor L3 Observations
  EO.ECMWF.DAT.REANALYSIS_UERRA_EUROPE_SINGLE_LEVELS  —  UERRA regional reanalysis for Europe on single levels from 1961 to 2019
  EO.MO.DAT.GLOBAL_ANALYSISFORECAST_PHY_001_024  —  Gl

In [5]:
# OR search: temperature or land
results = catalog.collection_search(q="temperature,land").collection_list()
print_results(results, "q='temperature,land'")

q='temperature,land': 161 collections found
  EO.ECMWF.DAT.ERA5_LAND_HOURLY  —  ERA5-Land hourly data from 1950 to present
  EO.ECMWF.DAT.ERA5_LAND_MONTHLY  —  ERA5-Land monthly averaged data from 1950 to present
  EO.EUM.DAT.MSG.LSA-LSTDE  —  Land Surface Temperature with Directional Effects - MSG
  ML.EUROSAT.DAT.LULC_10_GEOREF  —  A Novel Dataset for Land Use and Land Cover Classification
  EO.EUM.DAT.METOP.LSA-002  —  Daily Land Surface Temperature - Metop
  EO.CLMS.DAT.CORINE  —  CORINE Land Cover
  EO.GHSL.DAT.LAND  —  GHS-LAND R2022A - Land fraction as derived from Sentinel2 image composite (2018) and OSM data
  EO.ECMWF.DAT.REANALYSIS_UERRA_EUROPE_SINGLE_LEVELS  —  UERRA regional reanalysis for Europe on single levels from 1961 to 2019
  EO.MO.DAT.SST_GLO_SST_L4_NRT_OBSERVATIONS_010_001  —  Global Ocean OSTIA Sea Surface Temperature and Sea Ice Analysis
  EO.EUM.DAT.METOP.IASTHR011  —  IASI All Sky Temperature and Humidity Profiles - Climate Data Record Release 1.1 - Metop-A an

In [6]:
# Exact phrase
results = catalog.collection_search(q='"climate change"').collection_list()
print_results(results, "q='\"climate change\"'")

q='"climate change"': 57 collections found
  EO.ECMWF.DAT.D1.DT_CLIMATE.G1.CMIP6_HIST_ICON.R1  —  Climate Change Adaptation Digital Twin (Climate Adaptation DT) - Historical Simulation - ICON - Generation-1 - Realization-1
  EO.ECMWF.DAT.D1.DT_CLIMATE.G1.CMIP6_HIST_IFS-NEMO.R1  —  Climate Change Adaptation Digital Twin (Climate Adaptation DT) - Historical Simulation - IFS-NEMO - Generation-1 - Realization-1
  EO.ECMWF.DAT.D1.DT_CLIMATE.G1.HIGHRESMIP_CONT_IFS-FESOM.R1  —  Climate Change Adaptation Digital Twin (Climate Adaptation DT) - Control Simulation - IFS-FESOM - Generation-1 - Realization-1
  EO.ECMWF.DAT.D1.DT_CLIMATE.G1.HIGHRESMIP_CONT_IFS-NEMO.R1  —  Climate Change Adaptation Digital Twin (Climate Adaptation DT) - Control Simulation - IFS-NEMO - Generation-1 - Realization-1
  EO.ECMWF.DAT.D1.DT_CLIMATE.G1.SCENARIOMIP_SSP3-7.0_ICON.R1  —  Climate Change Adaptation Digital Twin (Climate Adaptation DT) - Future Projection - ICON - Generation-1 - Realization-1
  EO.ECMWF.DAT.D1.DT_

## 3. Browse by provider / information type

The collection ID convention (`EO.<PROVIDER>.DAT...`) lets us group the whole catalog by data provider. This gives a quick mental map of what kind of information lives in the Data Lake: ECMWF (models, reanalysis, Digital Twins), ESA (Sentinel satellites), EUM (EUMETSAT satellites), and so on.

In [7]:
providers = {}

for collection in collections:
    parts = collection.id.split(".")
    provider = parts[1] if len(parts) > 1 else "OTHER"
    providers.setdefault(provider, []).append(collection.id)

for provider, ids in sorted(providers.items()):
    print(f"{provider}: {len(ids)} collections")
    for collection_id in ids[:3]:
        print(f"  e.g. {collection_id}")
    print()

AERIS: 1 collections
  e.g. EO.AERIS.DAT.IAGOS

CLMS: 12 collections
  e.g. EO.CLMS.DAT.CORINE
  e.g. EO.CLMS.DAT.GLO.DMP300_V1
  e.g. EO.CLMS.DAT.GLO.FAPAR300_V1

DEM: 4 collections
  e.g. EO.DEM.DAT.COP-DEM_GLO-30-DGED
  e.g. EO.DEM.DAT.COP-DEM_GLO-30-DTED
  e.g. EO.DEM.DAT.COP-DEM_GLO-90-DGED

ECMWF: 87 collections
  e.g. EO.ECMWF.DAT.D1.DT_CLIMATE.G1.CMIP6_HIST_ICON.R1
  e.g. EO.ECMWF.DAT.D1.DT_CLIMATE.G1.CMIP6_HIST_IFS-NEMO.R1
  e.g. EO.ECMWF.DAT.D1.DT_CLIMATE.G1.HIGHRESMIP_CONT_IFS-FESOM.R1

ESA: 10 collections
  e.g. EO.ESA.DAT.SENTINEL-1.L1_GRD
  e.g. EO.ESA.DAT.SENTINEL-1.L1_SLC
  e.g. EO.ESA.DAT.SENTINEL-2.MSI.L1C

EUM: 116 collections
  e.g. EO.EUM.CM.METOP.ASCSZFR02
  e.g. EO.EUM.CM.METOP.ASCSZOR02
  e.g. EO.EUM.CM.METOP.ASCSZRR02

EUROSAT: 1 collections
  e.g. ML.EUROSAT.DAT.LULC_10_GEOREF

EUSTAT: 10 collections
  e.g. STAT.EUSTAT.DAT.AVAILABLE_BEDS_HOSPITALS_NUTS2
  e.g. STAT.EUSTAT.DAT.BATHING_SITES_WATER_QUALITY
  e.g. STAT.EUSTAT.DAT.GREENHOUSE_GAS_EMISSION_AGRICULTUR

Each collection also carries its own `keywords`, which describe the thematic content independently of the provider. Inspecting them helps to understand what a collection is about before using it.

In [ ]:
collection = catalog.get_collection("EO.EUM.CM.METOP.ASCSZFR02") # ID

print(f"ID:          {collection.id}")
print(f"Title:       {collection.title}")
print(f"Keywords:    {collection.keywords}")
print(f"Description: {collection.description[:300]}...")

ID:          EO.EUM.CM.METOP.ASCSZFR02
Title:       ASCAT Level 1 SZF Climate Data Record Release 2 - Metop
Keywords:    ['Climate', 'Fundamental Climate Data Record', '1B']
Description: Reprocessed L1B data from the Advanced Scatterometer (ASCAT) on METOP-A, resampled at full resolution (SZF). Normalized radar cross section (NRCS) of the Earth surface together with measurement time, location (latitude and longitude) and geometrical information (incidence and azimuth angles). The pr...


## 4. Filter by space and time

`collection_search` also accepts a bounding box and a datetime range. Only collections whose declared extent intersects the filter are returned.
This is particlarly useful to answer questions such as : "which datasets cover my study area in this period?"

In [9]:
# Bounding box around Paris: [west, south, east, north]
bbox = [2.2, 48.8, 2.4, 49.0]

results = catalog.collection_search(bbox=bbox).collection_list()
print(f"Collections covering the Paris area: {len(results)}\n")

# Combine space and time (Aapril 2026)
results = catalog.collection_search(
    bbox=bbox,
    datetime="2026-04-01T00:00:00Z/2026-04-30T00:00:00Z",
).collection_list()

print(f"Collections covering Paris with data in April 2026: {len(results)}\n")
for collection in results:
    print(f"  {collection.id}  —  {collection.title}")

Collections covering the Paris area: 294

Collections covering Paris with data in April 2026: 124

  EO.ECMWF.DAT.D1.DT_CLIMATE.G1.SCENARIOMIP_SSP3-7.0_ICON.R1  —  Climate Change Adaptation Digital Twin (Climate Adaptation DT) - Future Projection - ICON - Generation-1 - Realization-1
  EO.ECMWF.DAT.D1.DT_CLIMATE.G1.SCENARIOMIP_SSP3-7.0_IFS-FESOM.R1  —  Climate Change Adaptation Digital Twin (Climate Adaptation DT) - Future Projection - IFS-FESOM - Generation-1 - Realization-1
  EO.ECMWF.DAT.D1.DT_CLIMATE.G1.SCENARIOMIP_SSP3-7.0_IFS-NEMO.R1  —  Climate Change Adaptation Digital Twin (Climate Adaptation DT) - Future Projection - IFS-NEMO - Generation-1 - Realization-1
  EO.ECMWF.DAT.D1.DT_CLIMATE.G2.PROJECTIONS_SSP3-7.0_ICON.R1  —  Climate Change Adaptation Digital Twin (Climate Adaptation DT) - Future Projection - ICON - Generation-2 - Realization-1
  EO.ECMWF.DAT.D1.DT_CLIMATE.G2.PROJECTIONS_SSP3-7.0_IFS-FESOM.R1  —  Climate Change Adaptation Digital Twin (Climate Adaptation DT) - Futu

## 5. From collection to queryables

Once a collection is chosen, the next natural question is: *what filters can I use to search items inside it?* Those filters are called **queryables** and differ per collection (e.g. cloud cover for optical imagery, variable/pressure level for model data).

This is the bridge to the item-search step, which is out of scope here.

In [11]:
queryables = catalog.get_merged_queryables(
    collections=["EO.ECMWF.DAT.CAMS_GLOBAL_ATMOSHERIC_COMPO_FORECAST"]
)

print("Available search filters for EO.ECMWF.DAT.CAMS_GLOBAL_ATMOSHERIC_COMPO_FORECAST:\n")
for name, definition in queryables["properties"].items():
    print(f"  {name}  —  {definition.get('description', 'no description')}")

Available search filters for EO.ECMWF.DAT.CAMS_GLOBAL_ATMOSHERIC_COMPO_FORECAST:

  ecmwf:area  —  Select a sub-region of the available area by providing its limits on latitude and longitude
  ecmwf:data_format  —  Please select a format for the data files, the native format of MARS dataset is GRIB.
  ecmwf:date  —  Model base date. Note that dates older than 30 days will be slow access (see note above).
  ecmwf:leadtime_hour  —  Forecast lead time in hours
  ecmwf:model_level  —  Model level 1 is the top of the atmosphere. All levels changed on July 7 2019 00UTC when the number of levels was increased from 60 to 137. Model levels from this base time onwards are not comparable with those before.
  ecmwf:pressure_level  —  no description
  ecmwf:time  —  Model base time as HH:MM (UTC)
  ecmwf:type  —  no description
  ecmwf:variable  —  Multi level variables require the selection of a pressure or model level. Singe level variables do not. Slow access variables are archived on tape and w

## Summary

The HDA catalog can be explored without credentials through the STAC API. Depending on what you know beforehand, pick the method:

| I want to... | Method |
|---|---|
| See everything available | `catalog.get_collections()` |
| Find datasets about a topic | `collection_search(q="...")` |
| Map the catalog by provider | Group collection IDs by their `EO.<PROVIDER>` prefix |
| Find datasets covering an area/period | `collection_search(bbox=..., datetime=...)` |
| Know how to filter items in a collection | `catalog.get_merged_queryables(collections=[...])` |

In every case, the outcome is a **collection ID** — the key you need for the next step: searching and downloading items.

## Resources and references

- [DEDL documentation — Data Discovery](https://destine-data-lake-docs.data.destination-earth.eu/en/latest/dedl-discovery-and-data-access/Harmonized-Data-Access/API-Guide/Services-and-Data-Discovery.html)
- [DEDL documentation — HDA API Architecture](https://destine-data-lake-docs.data.destination-earth.eu/en/latest/dedl-discovery-and-data-access/Harmonized-Data-Access/API-Architecture/API-Architecture.html)
- [DestinE Data Portfolio (web UI)](https://hda.data.destination-earth.eu/ui/catalog)
- [pystac-client documentation](https://pystac-client.readthedocs.io/)
- Next steps in this lab: [HDA-PyStac-Client.ipynb](DestinE-DataLake-Lab/HDA/PySTAC/HDA-PyStac-Client.ipynb) for item search and access